In [17]:
import json
import time
from urllib.request import urlopen

WARM_SWITCHES_1_IP_ADDRESS = '169.254.10.10'
WARM_SWITCHES_2_IP_ADDRESS = '169.254.12.12'

def send_url_command(ip_address, cmd_to_send):
    url = f"http://{ip_address}/:{cmd_to_send}"
    for attempt in range(3):
        try:
            with urlopen(url, timeout=3) as response:
                pte_return = response.read()
            if len(pte_return) > 100:
                print(f"Error, command not found: {url}")
                return "Invalid Command!"
            return pte_return.decode('utf-8').strip()
        except Exception as network_err:
            if attempt == 2:
                print(f"Hardware Network Warning: No response from {ip_address}. Error: {network_err}")
                return "No Response!"
            time.sleep(0.05)

def get_warm_switches_state(ip_address):
    raw_reply = send_url_command(ip_address, "SWPORT?")
    if any(err in raw_reply for err in ["Error", "No Response"]):
        return {"status": "Disconnected or Error", "raw_reply": raw_reply}
    try:
        state_byte = int(raw_reply.strip())
        working_value = state_byte - 128
        return {
            "status": "Online",
            "raw_byte": state_byte,
            "A": working_value % 2,
            "B": (working_value // 2) % 2,
            "C": (working_value // 4) % 2,
            "D": (working_value // 8) % 16
        }
    except ValueError:
        return {"status": f"Malformed hardware response: {raw_reply}"}


def set_warm_switches_by_ip(ip_address, A, B, C, D):
    state_byte = 128 + int(A) + (2 * int(B)) + (4 * int(C)) + (8 * int(D))
    return send_url_command(ip_address, f"SETP={state_byte}")

def set_warm_switches(A1, B1, C1, D1, A2, B2, C2, D2):
    response_1 = set_all_switches_by_ip(WARM_SWITCHES_1_IP_ADDRESS, A1, B1, C1, D1)
    response_2 = set_all_switches_by_ip(WARM_SWITCHES_2_IP_ADDRESS, A2, B2, C2, D2)
    return response_1, response_2


In [14]:
set_all_switches_by_ip(WARM_SWITCHES_2_IP_ADDRESS,1,1,1,1)

'1'

In [20]:
set_all_switches_by_ip(WARM_SWITCHES_2_IP_ADDRESS,0,0,0,0)

'1'

In [21]:
get_warm_switches_state(WARM_SWITCHES_2_IP_ADDRESS)

{'status': 'Online', 'raw_byte': 128, 'A': 0, 'B': 0, 'C': 0, 'D': 0}